# Lab Day 1 — Notebook (PSEUDO-CODE)

> **Đây là khung pseudo-code.** Mọi ô có `# TODO` là phần bạn phải tự viết. Hãy giữ cấu trúc mục (Part 0–4) và
> viết dự đoán/nhận xét bằng chữ của bạn. Đọc `README.md`, `GUIDE.md`, `RUBRIC.md` trước.
>
> **Quy trình làm bài**
> 1. `git pull` repo, tạo thư mục nộp: copy cả thư mục `code/` vào `submission_<MSSV>/code/`.
> 2. Mở `submission_<MSSV>/code/lab.ipynb` và làm việc **tại đó** (nên `REPO_ROOT = "../.."` đúng và ảnh/kết quả ghi vào `submission_<MSSV>/figures`, `results`).
> 3. Ở Colab: chạy ô clone repo bên dưới, rồi làm tương tự (đặt `REPO_ROOT` cho đúng).
> 4. Cuối cùng: `Restart & Run All` phải chạy hết không lỗi và giữ nguyên output.

In [ ]:
# Colab bootstrap: mount Drive and run from the repo's code/ folder so REPO_ROOT=".." and OUT_DIR=".." resolve
import os, sys
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB: from google.colab import drive; drive.mount("/content/drive")
if IN_COLAB: os.chdir("/content/drive/MyDrive/K4-Track4-Day1-Neuralnetwork/code")
if os.getcwd() not in sys.path: sys.path.insert(0, os.getcwd())
%load_ext autoreload
%autoreload 2
print(os.getcwd())

In [ ]:
# ===== Cấu hình đường dẫn và môi trường =====
import os, sys, json, time, subprocess, math
import numpy as np, torch
import torch.nn.functional as F
import matplotlib.pyplot as plt

REPO_ROOT = ".."        # thư mục gốc repo (chứa data/ và scripts/), tính từ code/
OUT_DIR   = ".."        # nơi ghi figures/, results/, experiments.xlsx, predictions_eval.csv (= gốc repo)

device = "cuda" if torch.cuda.is_available() else ("mps" if torch.backends.mps.is_available() else "cpu")
print(f"torch {torch.__version__} | device = {device}" + (f" | GPU = {torch.cuda.get_device_name(0)}" if device == "cuda" else ""))
os.makedirs(f"{OUT_DIR}/figures", exist_ok=True)
os.makedirs(f"{OUT_DIR}/results", exist_ok=True)

from data import prepare_data
from model import MLP, EXPECTED_PARAMS, count_params, init_weights, activation_stats
from optimizer import build_optimizer, clip_gradients
from train import DEFAULT_CFG, set_seed, evaluate, predict, run_experiment, final_eval
from plots import plot_run, plot_compare
from results_table import save_result, load_results, to_row, write_xlsx

## Part 0 — Dữ liệu
Chia sẵn bằng metadata: chạy `scripts/split_data.py` từ thư mục gốc repo (tạo `data/processed/train.npz`, `eval.npz`).
Sau đó tách validation từ train, chuẩn hoá bằng thống kê của train, đưa lên device.

In [ ]:
# Part 0: tạo train.npz / eval.npz từ metadata (script tự kiểm tra tính toàn vẹn; chạy lại cho kết quả y hệt)
r = subprocess.run([sys.executable, "scripts/split_data.py"], cwd=REPO_ROOT, check=True, capture_output=True,
                   text=True, encoding="utf-8", env={**os.environ, "PYTHONIOENCODING": "utf-8"})
print(r.stdout)

# Tách val 20% từ train (phân tầng, seed 42), chuẩn hoá 10 cột số bằng mean/std của X_tr, đưa lên device
data = prepare_data(device, val_fraction=0.2, seed=42, processed_dir=f"{REPO_ROOT}/data/processed")
assert (len(data["X_tr"]), len(data["X_val"]), len(data["X_eval"])) == (371_847, 92_962, 116_203)

num = data["X_tr"][:, :10]
print("\nmean 10 cột số trên X_tr:", np.round(num.mean(0).cpu().numpy(), 4))
print("std  10 cột số trên X_tr:", np.round(num.std(0).cpu().numpy(), 4))
assert num.mean(0).abs().max() < 1e-3 and (num.std(0) - 1).abs().max() < 1e-3
assert set(torch.unique(data["X_tr"][:, 10:]).tolist()) <= {0.0, 1.0}   # 44 cột nhị phân giữ nguyên

**Nhận xét Part 0:** val được tách **từ train** (20%, phân tầng, seed 42), nên tỉ lệ lớp ở train/val/eval gần như trùng nhau.
Mean/std chỉ tính trên `X_tr`. Nếu tính cả trên val/eval, thống kê chuẩn hoá đã "nhìn thấy" chính dữ liệu dùng để đo (rò rỉ thông tin), nên điểm val/eval không còn đo đúng khả năng trên dữ liệu mới.
`X_eval` chỉ được biến đổi bằng mean/std đó và không tham gia chọn cấu hình hay dừng sớm. Lô cuối của mỗi epoch được giữ lại dù nhỏ hơn (371 847 = 726·512 + 135).
Mốc thấp nhất phải vượt: luôn đoán lớp đa số → accuracy ≈ 0,4876 trên val.

## Part 1 — Model và kiểm tra "sức khoẻ" ban đầu
Quy định: `M-base` = `54 → 256 → 128 → 7`, **47 879 tham số**, logits `(B, 7)`, không softmax trong model.

In [ ]:
# Part 1a/1b: M-base, số tham số và shape sau từng lớp
set_seed(1)
model = MLP(hidden=(256, 128), dropout=0.0, init="he").to(device)
n_params = sum(p.numel() for p in model.parameters())
print(model)
print("số tham số:", n_params)
assert n_params == count_params(model) == EXPECTED_PARAMS[(256, 128)] == 47_879

with torch.no_grad():
    h = torch.randn(8, 54, device=device)
    print(f"{'input':>8} -> {tuple(h.shape)}")
    for layer in model.net:
        h = layer(h)
        print(f"{type(layer).__name__:>8} -> {tuple(h.shape)}")
assert h.shape == (8, 7)

In [ ]:
# Part 1c-1: loss bước 0 trên val (eval mode, chưa cập nhật bước nào) phải ≈ ln 7
model.eval()
with torch.no_grad():
    logits0 = model(data["X_val"])
    step0_loss = F.cross_entropy(logits0, data["y_val"]).item()
print(f"loss bước 0 trên val = {step0_loss:.4f} | ln 7 = {math.log(7):.4f} | chênh = {step0_loss - math.log(7):+.4f}")
print(f"std của logit bước 0 = {logits0.std().item():.3f}")
print("std kích hoạt sau mỗi Linear (z1, z2, logits), lô val 4096 mẫu:",
      [round(s, 3) for s in activation_stats(model, data["X_val"][:4096])])

In [ ]:
# Part 1c-2: quá khớp 20 mẫu (dropout tắt). Loss phải về gần 0 và accuracy 100% trên 20 mẫu đó
set_seed(1)
idx = torch.randperm(len(data["X_tr"]), generator=torch.Generator().manual_seed(0))[:20].to(device)
x20, y20 = data["X_tr"][idx], data["y_tr"][idx]
tiny = MLP(hidden=(256, 128), dropout=0.0, init="he").to(device)
opt = torch.optim.Adam(tiny.parameters(), lr=1e-2)
losses = []
for step in range(300):
    loss = F.cross_entropy(tiny(x20), y20)
    opt.zero_grad(set_to_none=True)
    loss.backward()
    opt.step()
    losses.append(loss.item())
with torch.no_grad():
    acc20 = (tiny(x20).argmax(1) == y20).float().mean().item()
print(f"nhãn của 20 mẫu: {torch.bincount(y20, minlength=7).tolist()}")
print(f"loss bước 0 = {losses[0]:.4f}, bước {len(losses) - 1} = {losses[-1]:.2e}; accuracy trên 20 mẫu = {acc20:.0%}")

plt.figure(figsize=(6, 3.5))
plt.semilogy(losses)
plt.xlabel("bước"); plt.ylabel("cross-entropy (log)")
plt.title("Quá khớp 20 mẫu: M-base, Adam lr=1e-2, dropout 0")
plt.grid(alpha=0.3); plt.tight_layout(); plt.show()
assert losses[-1] < 1e-2 and acc20 == 1.0

In [ ]:
# Part 1d: sau một lần backward, mọi tham số phải có gradient khác None và khác 0
model.train()
model.zero_grad(set_to_none=True)
F.cross_entropy(model(data["X_tr"][:512]), data["y_tr"][:512]).backward()
for name, (pname, p) in zip(["W1", "b1", "W2", "b2", "W3", "b3"], model.named_parameters()):
    assert p.grad is not None, f"{name}: grad là None"
    gn = p.grad.norm().item()
    print(f"{name}  {pname:<12} {str(tuple(p.shape)):<11} grad norm = {gn:.4e}")
    assert gn > 0, f"{name}: grad bằng 0"

**Nhận xét Part 1 (tự viết):** loss bước 0 đo được = ___, so với ln 7 = 1.946 vì sao ...; quá khớp 20 mẫu: ...

## Part 2 — Pipeline và baseline
`run_experiment(cfg, data)` nhận một dict cấu hình và trả về lịch sử + tóm tắt. Baseline: SGD+momentum 0.9, CE, He init, batch 512, 20 epoch.

In [ ]:
# ===== Tiện ích cho Part 2-4: chạy một cfg -> lưu results/<exp_id>.json -> vẽ figures/<exp_id>.png =====
import pandas as pd
from IPython.display import Image, display

USE_CACHE = True   # Colab hay ngắt kết nối: exp_id đã có JSON với ĐÚNG cfg này thì đọc lại thay vì chạy lại
QUICK = os.environ.get("LAB_QUICK") == "1"   # chỉ để thử notebook nhanh (1 epoch, 20 000 mẫu train); mặc định tắt
if QUICK:
    data = {**data, "X_tr": data["X_tr"][:20_000], "y_tr": data["y_tr"][:20_000]}
RESULTS = {}       # exp_id -> result


def _norm_cfg(c):
    return json.loads(json.dumps({**DEFAULT_CFG, **c, "hidden": list(c.get("hidden", DEFAULT_CFG["hidden"]))}))


def run(cfg, force=False, show=False):
    """Chạy (hoặc đọc lại từ JSON) một thí nghiệm, lưu JSON + ảnh, ghi vào RESULTS."""
    cfg = {**DEFAULT_CFG, **cfg}
    if QUICK:
        cfg["epochs"] = 1
    path = f"{OUT_DIR}/results/{cfg['exp_id']}.json"
    cached = None
    if USE_CACHE and not force and os.path.exists(path):
        with open(path, encoding="utf-8") as f:
            cached = json.load(f)
        if _norm_cfg(cached["cfg"]) != _norm_cfg(cfg):
            print(f"[{cfg['exp_id']}] cfg khác file đã lưu -> chạy lại")
            cached = None
    if cached is not None:
        cached["cfg"]["hidden"] = tuple(cached["cfg"]["hidden"])
        res, s = cached, cached["summary"]
        print(f"[{cfg['exp_id']}] đọc lại {path}: best epoch {s['best_epoch']} | val acc {s['val_acc']:.4f} | "
              f"val macro-F1 {s['val_macro_f1']:.4f}")
    else:
        res = run_experiment(cfg, data)
        save_result(res, f"{OUT_DIR}/results")
    plot_run(res, f"{OUT_DIR}/figures/{cfg['exp_id']}.png")
    RESULTS[cfg["exp_id"]] = res
    if show:
        display(Image(f"{OUT_DIR}/figures/{cfg['exp_id']}.png"))
    return res


def f1(exp_id):
    return RESULTS[exp_id]["summary"]["val_macro_f1"]


def report(ids):
    """Bảng tóm tắt; khi đã có baseline nhiều seed thì thêm chênh lệch val macro-F1 so với trung bình baseline và 2σ."""
    rows = []
    for i in ids:
        s = RESULTS[i]["summary"]
        row = {"exp_id": i, "step0_loss": s["step0_loss"], "best_epoch": s["best_epoch"],
               "best_val_loss": s["best_val_loss"], "final_train_loss": s["final_train_loss"],
               "final_val_loss": s["final_val_loss"], "val_acc": s["val_acc"], "val_macro_f1": s["val_macro_f1"],
               "s/epoch": s["time_per_epoch_s"], "peak_mem_MB": s["peak_mem_MB"],
               "Δmem_MB": s["peak_mem_MB"] - s["base_mem_MB"], "diverged": s["diverged"]}
        if "NOISE_F1" in globals():
            row["Δf1_vs_base"] = s["val_macro_f1"] - BASE_F1_MEAN
            row["vượt 2σ?"] = "Có" if abs(row["Δf1_vs_base"]) > NOISE_F1 else "Không"
        rows.append(row)
    display(pd.DataFrame(rows).set_index("exp_id").round(4))

In [ ]:
# Part 2: chọn lr cho baseline (SGD+momentum 0.9, M-base, He, batch 512, 20 epoch, seed 1) CHỈ bằng VAL macro-F1.
# Mỗi lần quét cũng là thí nghiệm của chủ đề optimizer (SGD+momentum ở 4 lr): mỗi lần một dòng trong bảng + một ảnh.
LR_GRID = {"sgd_momentum": [0.01, 0.03, 0.1, 0.3]}
SHORT = {"sgd": "sgd", "sgd_momentum": "sgdm", "adam": "adam", "adamw": "adamw"}


def opt_id(opt, lr):
    return f"opt-{SHORT[opt]}-lr{lr:g}"


def best_lr(opt):
    """lr có val macro-F1 cao nhất trong lưới của opt; cảnh báo nếu nó nằm ở biên lưới (nên thử thêm lr ngoài biên)."""
    lrs = LR_GRID[opt]
    lr = max(lrs, key=lambda v: f1(opt_id(opt, v)))
    if lr in (min(lrs), max(lrs)):
        print(f"chú ý: lr tốt nhất của {opt} ({lr:g}) nằm ở biên lưới {lrs} -> nên thử thêm một lr ngoài biên")
    return lr


for lr in LR_GRID["sgd_momentum"]:
    run({"exp_id": opt_id("sgd_momentum", lr), "group": "optimizer", "lr": lr,
         "description": f"SGD+momentum 0.9, lr={lr:g} (quét lr để chọn lr baseline)"})
report([opt_id("sgd_momentum", lr) for lr in LR_GRID["sgd_momentum"]])
BASE_LR = best_lr("sgd_momentum")
BASE_CFG = {**DEFAULT_CFG, "lr": BASE_LR}
print(f"lr baseline chọn theo val macro-F1: {BASE_LR:g}")

In [ ]:
# Baseline với 3 seed để đo độ nhiễu: chênh lệch val macro-F1 nhỏ hơn 2σ không phải là bằng chứng.
BASE_IDS = ["base-s1", "base-s2", "base-s3"]
for seed, exp_id in zip((1, 2, 3), BASE_IDS):
    run({**BASE_CFG, "exp_id": exp_id, "group": "baseline", "seed": seed,
         "description": f"Baseline M-base, seed {seed}"}, show=(seed == 1))

f1s = np.array([f1(i) for i in BASE_IDS])
accs = np.array([RESULTS[i]["summary"]["val_acc"] for i in BASE_IDS])
BASE_F1_MEAN, BASE_F1_STD = f1s.mean(), f1s.std(ddof=1)   # ddof=1 giống STDEV ở sheet Seeds
NOISE_F1 = 2 * BASE_F1_STD
report(BASE_IDS)
print(f"val macro-F1 = {BASE_F1_MEAN:.4f} ± {BASE_F1_STD:.4f} (3 seed) -> ngưỡng nhiễu 2σ = {NOISE_F1:.4f}")
print(f"val acc      = {accs.mean():.4f} ± {accs.std(ddof=1):.4f} (mốc đoán đa số 0.4876)")
assert accs.min() > 0.4876, "baseline không vượt mốc đoán đa số -> có lỗi"

plot_compare([RESULTS[i] for i in BASE_IDS], ["train_loss", "val_loss", "val_macro_f1"],
             f"{OUT_DIR}/figures/compare_baseline.png", "Baseline M-base: 3 seed")
display(Image(f"{OUT_DIR}/figures/compare_baseline.png"))

**Nhận xét baseline (tự viết):** hình dạng đường cong (còn giảm? bắt đầu quá khớp?), val_acc so với mốc "đoán đa số", độ nhiễu giữa các seed ...

## Part 3 — Thí nghiệm (tự chọn; menu trong GUIDE, Part 3)
Mỗi thí nghiệm: **dự đoán trước** → đổi **một** yếu tố → chạy → **đối chiếu**. Lặp lại khối dưới cho từng thí nghiệm bạn chọn.
Chủ đề: loss · optimizer · hparam · dropout · clipping · amp · init.

### Chủ đề 1 — Hàm mất mát (`loss`): CE → MSE trên one-hot
**Dự đoán trước (viết TRƯỚC khi chạy):** ...

In [ ]:
# Chủ đề 1 — loss: chỉ đổi CE -> MSE (cùng lr, seed, 20 epoch). MSE = F.mse_loss(logits, one_hot(y)):
# trung bình trên B×7 phần tử, không có hệ số 1/2 (= nn.MSELoss mặc định). So acc/macro-F1, KHÔNG so giá trị loss.
run({**BASE_CFG, "exp_id": "loss-mse", "group": "loss", "loss": "mse",
     "description": "MSE trên one-hot thay cho CE",
     "notes": "MSE trung bình trên B×7 phần tử, không hệ số 1/2; khác thang đo CE nên chỉ so acc/F1"})
report(["base-s1", "loss-mse"])
plot_compare([RESULTS["base-s1"], RESULTS["loss-mse"]], ["val_acc", "val_macro_f1", "grad_norm"],
             f"{OUT_DIR}/figures/compare_loss.png", "CE (base-s1) vs MSE: so acc/F1, không so giá trị loss")
display(Image(f"{OUT_DIR}/figures/compare_loss.png"))

**Đối chiếu sau khi chạy:** khớp / khác dự đoán? vì sao (cơ chế)? chênh lệch có vượt 2σ không? ...

### Chủ đề 2 — Bộ tối ưu (`optimizer`): SGD, SGD+momentum, Adam, AdamW, mỗi bộ ≥ 3 lr
**Dự đoán trước (viết TRƯỚC khi chạy):** ...

In [ ]:
# Chủ đề 2 — optimizer: mỗi bộ ≥ 3 lr (cách nhau ~3 lần), so ở lr tốt nhất của mỗi bộ (theo VAL).
# SGD+momentum đã quét ở Part 2. SGD không momentum cần lr lớn hơn (~1/(1-μ) = 10 lần) mới có cùng bước hiệu dụng.
# Adam/AdamW: betas=(0.9, 0.999), eps=1e-8; AdamW weight_decay=0.01 (tách riêng).
LR_GRID.update({"sgd": [0.1, 0.3, 1.0], "adam": [3e-4, 1e-3, 3e-3, 1e-2], "adamw": [1e-3, 3e-3, 1e-2]})
NAMES = {"sgd": "SGD (không momentum)", "adam": "Adam", "adamw": "AdamW wd=0.01"}
for opt in ("sgd", "adam", "adamw"):
    for lr in LR_GRID[opt]:
        run({**BASE_CFG, "exp_id": opt_id(opt, lr), "group": "optimizer", "optimizer": opt, "lr": lr,
             "weight_decay": 0.01 if opt == "adamw" else 0.0, "description": f"{NAMES[opt]}, lr={lr:g}"})
OPT_IDS = {opt: [opt_id(opt, lr) for lr in lrs] for opt, lrs in LR_GRID.items()}
BEST_OPT = {opt: opt_id(opt, best_lr(opt)) for opt in LR_GRID}
report([i for ids in OPT_IDS.values() for i in ids])
print("lr tốt nhất của mỗi optimizer (theo val macro-F1):", BEST_OPT)
report(list(BEST_OPT.values()))

fig, ax = plt.subplots(figsize=(6.5, 4))
for opt, lrs in LR_GRID.items():
    ax.plot(lrs, [f1(opt_id(opt, lr)) for lr in lrs], "o-", label=opt)
ax.axhspan(BASE_F1_MEAN - NOISE_F1, BASE_F1_MEAN + NOISE_F1, color="gray", alpha=0.25, label="baseline ± 2σ")
ax.set_xscale("log"); ax.set_xlabel("lr"); ax.set_ylabel("val macro-F1 (best epoch)")
ax.set_title("Độ nhạy theo lr của từng optimizer"); ax.grid(alpha=0.3); ax.legend()
fig.savefig(f"{OUT_DIR}/figures/compare_optimizer_lr.png", dpi=110, bbox_inches="tight"); plt.show()
plot_compare([RESULTS[i] for i in BEST_OPT.values()], ["val_loss", "val_macro_f1"],
             f"{OUT_DIR}/figures/compare_optimizer.png", "Mỗi optimizer ở lr tốt nhất (theo val)")
display(Image(f"{OUT_DIR}/figures/compare_optimizer.png"))

**Đối chiếu sau khi chạy:** khớp / khác dự đoán? vì sao (cơ chế)? chênh lệch có vượt 2σ không? ...

### Chủ đề 3 — Hyper-parameter (`hparam`): batch size, độ rộng, độ sâu
**Dự đoán trước (viết TRƯỚC khi chạy):** ...

In [ ]:
# Chủ đề 3 — hyper-parameter: batch size (cùng 20 epoch nhưng SỐ BƯỚC cập nhật khác nhau), độ rộng, độ sâu.
def steps(batch):
    return math.ceil(len(data["X_tr"]) / batch)


HP = [
    dict(exp_id="hp-batch128", batch=128, description="batch 128 (lr giữ nguyên)"),
    dict(exp_id="hp-batch2048", batch=2048, description="batch 2048 (lr giữ nguyên)"),
    dict(exp_id="hp-batch2048-lrx4", batch=2048, lr=4 * BASE_LR, scheduler="warmup", warmup_steps=steps(2048),
         description="batch 2048, lr ×4 (quy tắc tăng lr theo lô) + khởi động tuyến tính 1 epoch",
         notes="đổi có chủ đích batch ×4 và lr ×4, kèm warmup 1 epoch"),
    dict(exp_id="hp-wide", hidden=(512, 256), description="M-wide 54-512-256-7 (161 287 tham số)"),
    dict(exp_id="hp-deep", hidden=(256, 128, 64), description="M-deep 54-256-128-64-7 (55 687 tham số)"),
]
for c in HP:
    run({**BASE_CFG, "group": "hparam", **c})
HP_IDS = [c["exp_id"] for c in HP]
report(["base-s1"] + HP_IDS)
for b in (128, 512, 2048):
    print(f"batch {b:>4}: {steps(b):>4} bước/epoch -> {20 * steps(b):>6} bước trong 20 epoch")

plot_compare([RESULTS[i] for i in ("base-s1", "hp-batch128", "hp-batch2048", "hp-batch2048-lrx4")],
             ["val_loss", "val_macro_f1", "epoch_time_s"], f"{OUT_DIR}/figures/compare_hparam_batch.png",
             "Batch size (cùng 20 epoch, khác số bước)")
plot_compare([RESULTS[i] for i in ("base-s1", "hp-wide", "hp-deep")], ["train_loss", "val_loss", "val_macro_f1"],
             f"{OUT_DIR}/figures/compare_hparam_size.png", "Độ rộng / độ sâu: M-base vs M-wide vs M-deep")
display(Image(f"{OUT_DIR}/figures/compare_hparam_batch.png"), Image(f"{OUT_DIR}/figures/compare_hparam_size.png"))

**Đối chiếu sau khi chạy:** khớp / khác dự đoán? vì sao (cơ chế)? chênh lệch có vượt 2σ không? ...

### Chủ đề 4 — Dropout (`dropout`): q ∈ {0,1; 0,3; 0,5}
**Dự đoán trước (viết TRƯỚC khi chạy):** ...

In [ ]:
# Chủ đề 4 — dropout q (xác suất TẮT) sau mỗi ReLU ẩn. Train loss đo ở eval() nên so được với val.
DROP_IDS = []
for q in (0.1, 0.3, 0.5):
    run({**BASE_CFG, "exp_id": f"drop-{q:g}", "group": "dropout", "dropout": q,
         "description": f"dropout q={q:g} sau mỗi ReLU ẩn"})
    DROP_IDS.append(f"drop-{q:g}")
report(["base-s1"] + DROP_IDS)
for i in ["base-s1"] + DROP_IDS:
    s = RESULTS[i]["summary"]
    print(f"{i:>8}: khoảng cách val - train loss ở epoch cuối = {s['final_val_loss'] - s['final_train_loss']:+.4f}")
plot_compare([RESULTS[i] for i in ["base-s1"] + DROP_IDS], ["train_loss", "val_loss", "val_macro_f1"],
             f"{OUT_DIR}/figures/compare_dropout.png", "Dropout q ∈ {0, 0.1, 0.3, 0.5}")
display(Image(f"{OUT_DIR}/figures/compare_dropout.png"))

**Đối chiếu sau khi chạy:** khớp / khác dự đoán? vì sao (cơ chế)? chênh lệch có vượt 2σ không? ...

### Chủ đề 5 — Cắt gradient (`clipping`)
**Dự đoán trước (viết TRƯỚC khi chạy):** ...

In [ ]:
# Chủ đề 5 — clipping: chọn c từ phân phối grad_norm (trước clip) của baseline, để clip THỰC SỰ kích hoạt.
s = RESULTS["base-s1"]["summary"]
print(f"grad_norm mỗi bước của base-s1: p50 {s['grad_norm_p50']:.3f} | p90 {s['grad_norm_p90']:.3f} | "
      f"p99 {s['grad_norm_p99']:.3f} | max {s['grad_norm_max']:.3f}")
CLIP_C = float(f"{s['grad_norm_p50']:.2g}")      # ≈ trung vị -> clip ở khoảng một nửa số bước
CLIP_IDS = [f"clip-{CLIP_C:g}"]
run({**BASE_CFG, "exp_id": CLIP_IDS[0], "group": "clipping", "clip_norm": CLIP_C,
     "description": f"clip_norm c={CLIP_C:g} (≈ trung vị grad_norm của baseline), lr baseline"})
print("tỉ lệ bước bị clip mỗi epoch:", np.round(RESULTS[CLIP_IDS[0]]["history"]["clip_frac"], 2).tolist())

# Phản chứng: tăng lr tới khi KHÔNG clip thì mất ổn định (NaN hoặc val macro-F1 kém hẳn), rồi chạy cùng lr đó CÓ clip.
for mult in (10, 30, 100):
    HIGH_LR = BASE_LR * mult
    exp_id = f"clip-none-lr{HIGH_LR:g}"
    r = run({**BASE_CFG, "exp_id": exp_id, "group": "clipping", "lr": HIGH_LR,
             "description": f"lr ×{mult} = {HIGH_LR:g}, KHÔNG clip (tìm lr gây mất ổn định)"})
    CLIP_IDS.append(exp_id)
    if r["summary"]["diverged"] or f1(exp_id) < BASE_F1_MEAN - 0.05:
        break
exp_id = f"clip-{CLIP_C:g}-lr{HIGH_LR:g}"
run({**BASE_CFG, "exp_id": exp_id, "group": "clipping", "lr": HIGH_LR, "clip_norm": CLIP_C,
     "description": f"lr ×{mult} = {HIGH_LR:g}, CÓ clip c={CLIP_C:g}"})
CLIP_IDS.append(exp_id)
report(["base-s1"] + CLIP_IDS)
plot_compare([RESULTS[i] for i in ["base-s1"] + CLIP_IDS], ["val_loss", "val_macro_f1", "grad_norm_max"],
             f"{OUT_DIR}/figures/compare_clipping.png", f"Clipping c={CLIP_C:g}: lr baseline và lr cao có/không clip")
display(Image(f"{OUT_DIR}/figures/compare_clipping.png"))

**Đối chiếu sau khi chạy:** khớp / khác dự đoán? vì sao (cơ chế)? chênh lệch có vượt 2σ không? ...

### Chủ đề 6 — Mixed precision (`amp`): FP16, BF16
**Dự đoán trước (viết TRƯỚC khi chạy):** ...

In [ ]:
# Chủ đề 6 — mixed precision. Tham số vẫn FP32; autocast chỉ hạ độ chính xác của phép toán trong forward + loss.
# FP16 dùng GradScaler (scaler.unscale_ trước khi đo/clip grad_norm); BF16 không cần GradScaler.
if device == "cuda":
    cap = torch.cuda.get_device_capability()
    try:
        bf16_native = torch.cuda.is_bf16_supported(including_emulation=False)
    except TypeError:
        bf16_native = cap >= (8, 0)
    print(f"GPU {torch.cuda.get_device_name(0)} | compute capability {cap[0]}.{cap[1]} | BF16 phần cứng: {bf16_native}")
else:
    bf16_native = False
    print("không có GPU: autocast chạy trên CPU, thời gian/bộ nhớ không đại diện cho GPU")
AMP_NOTES = {"fp16": "autocast fp16 + GradScaler; unscale_ mọi bước trước khi đo grad_norm",
             "bf16": "autocast bf16, không GradScaler" + ("" if bf16_native else
                     "; GPU không có phần cứng BF16 (T4 = sm75) nên chạy bằng giả lập, không đại diện tốc độ BF16")}
AMP_IDS = []
for prec in ("fp16", "bf16"):
    try:
        run({**BASE_CFG, "exp_id": f"amp-{prec}", "group": "amp", "precision": prec,
             "description": f"mixed precision {prec} (autocast)", "notes": AMP_NOTES[prec]})
        AMP_IDS.append(f"amp-{prec}")
    except Exception as e:      # ví dụ GPU/PyTorch không hỗ trợ: ghi lý do vào báo cáo
        print(f"amp-{prec} không chạy được: {type(e).__name__}: {e}")
report(["base-s1"] + AMP_IDS)
if "amp-fp16" in RESULTS:
    print("số bước GradScaler bỏ qua (gradient inf/NaN) mỗi epoch:", RESULTS["amp-fp16"]["history"]["amp_skipped"])
plot_compare([RESULTS[i] for i in ["base-s1"] + AMP_IDS], ["val_loss", "val_macro_f1", "epoch_time_s"],
             f"{OUT_DIR}/figures/compare_amp.png", "FP32 vs FP16 vs BF16")
display(Image(f"{OUT_DIR}/figures/compare_amp.png"))

**Đối chiếu sau khi chạy:** khớp / khác dự đoán? vì sao (cơ chế)? chênh lệch có vượt 2σ không? ...

### Chủ đề 7 — Khởi tạo (`init`): zeros, normal(0,01), xavier so với He
**Dự đoán trước (viết TRƯỚC khi chạy):** ...

In [ ]:
# Chủ đề 7 — khởi tạo (bias = 0). xavier = nn.init.xavier_normal_: Var[W] = 2/(n_vào + n_ra); he: Var[W] = 2/n_vào.
# Bước 0: độ lệch chuẩn kích hoạt sau mỗi Linear (lô val 4096 mẫu) và loss bước 0 cho từng cách khởi tạo.
rows = []
for init in ("he", "xavier", "normal", "zeros", "default"):
    set_seed(1)
    m0 = MLP(hidden=(256, 128), init=init).to(device).eval()
    with torch.no_grad():
        l0 = F.cross_entropy(m0(data["X_val"]), data["y_val"]).item()
    z1, z2, z3 = activation_stats(m0, data["X_val"][:4096])
    rows.append({"init": init, "std z1": z1, "std z2": z2, "std logits": z3, "loss bước 0": l0})
display(pd.DataFrame(rows).set_index("init").round(4))

INIT_IDS = []
for init in ("zeros", "normal", "xavier"):
    run({**BASE_CFG, "exp_id": f"init-{init}", "group": "init", "init": init,
         "description": f"khởi tạo {init} thay cho He (bias = 0)"})
    INIT_IDS.append(f"init-{init}")
report(["base-s1"] + INIT_IDS)

# zeros: mọi kích hoạt ẩn = ReLU(0) = 0 và W3 = 0, nên sau một backward chỉ b3 có gradient khác 0
# -> mạng chỉ học được tần suất lớp (luôn đoán lớp đa số), các nơ-ron cùng lớp mãi đối xứng
set_seed(1)
mz = MLP(hidden=(256, 128), init="zeros").to(device)
F.cross_entropy(mz(data["X_tr"][:512]), data["y_tr"][:512]).backward()
print("init=zeros, grad norm từng tham số sau 1 backward:",
      {n: round(p.grad.norm().item(), 6) for n, p in mz.named_parameters()})
plot_compare([RESULTS[i] for i in ["base-s1"] + INIT_IDS], ["train_loss", "val_loss", "val_macro_f1"],
             f"{OUT_DIR}/figures/compare_init.png", "Khởi tạo: he (base-s1) vs xavier vs normal(0.01) vs zeros")
display(Image(f"{OUT_DIR}/figures/compare_init.png"))

**Đối chiếu sau khi chạy:** khớp / khác dự đoán? vì sao (cơ chế)? chênh lệch có vượt 2σ không? ...

### Cấu hình cuối cùng — chọn CHỈ bằng val
Kết hợp bộ tối ưu + lr tốt nhất (theo val, trong mọi lần chạy chủ đề optimizer) với M-base / M-wide và dropout {0; 0,1}, huấn luyện 60 epoch với lr giảm theo cosine.
Ứng viên tốt nhất theo val macro-F1 được chạy thêm 2 seed. Tập eval chỉ được dùng ở Part 4, sau khi đã chốt cấu hình.

**Dự đoán trước (viết TRƯỚC khi chạy):** ...

In [ ]:
best_opt_id = max([i for ids in OPT_IDS.values() for i in ids], key=f1)
oc = RESULTS[best_opt_id]["cfg"]
FINAL_BASE = {**BASE_CFG, "group": "final", "optimizer": oc["optimizer"], "lr": oc["lr"],
              "weight_decay": oc["weight_decay"], "epochs": 60, "scheduler": "cosine"}
print(f"optimizer + lr tốt nhất theo val: {best_opt_id} (val macro-F1 {f1(best_opt_id):.4f})")

CANDS = []
for size, hidden in (("base", (256, 128)), ("wide", (512, 256))):
    for q in (0.0, 0.1):
        exp_id = f"final-{size}-d{q:g}-s1"
        run({**FINAL_BASE, "exp_id": exp_id, "hidden": hidden, "dropout": q, "seed": 1,
             "description": f"ứng viên cuối: {oc['optimizer']} lr={oc['lr']:g}, {size}, dropout {q:g}, 60 epoch, cosine"})
        CANDS.append(exp_id)
report(CANDS)
FINAL_ID = max(CANDS, key=f1)
FINAL_CFG = RESULTS[FINAL_ID]["cfg"]
print("cấu hình cuối (theo val macro-F1):", FINAL_ID)

FINAL_IDS = [FINAL_ID]
for seed in (2, 3):
    exp_id = FINAL_ID.replace("-s1", f"-s{seed}")
    run({**FINAL_CFG, "exp_id": exp_id, "seed": seed, "description": f"{FINAL_CFG['description']}, seed {seed}"})
    FINAL_IDS.append(exp_id)
report(FINAL_IDS)
ff = np.array([f1(i) for i in FINAL_IDS])
print(f"cấu hình cuối: val macro-F1 {ff.mean():.4f} ± {ff.std(ddof=1):.4f} (3 seed) | "
      f"baseline {BASE_F1_MEAN:.4f} ± {BASE_F1_STD:.4f} | chênh {ff.mean() - BASE_F1_MEAN:+.4f}")
plot_compare([RESULTS[i] for i in ["base-s1"] + CANDS + FINAL_IDS[1:]], ["val_loss", "val_macro_f1"],
             f"{OUT_DIR}/figures/compare_final.png", "Ứng viên cấu hình cuối (val) so với baseline")
display(Image(f"{OUT_DIR}/figures/compare_final.png"))

**Đối chiếu sau khi chạy:** khớp / khác dự đoán? vì sao (cơ chế)? chênh lệch có vượt 2σ không? ...

## Part 4 — Đánh giá cuối trên eval, bảng và nộp bài
**Cách đánh giá:** chọn cấu hình CHỈ bằng val → chạy final model → dự đoán trên **toàn bộ** tập eval → ghi `predictions_eval.csv`
→ `python scripts/evaluate.py --pred ...` tính accuracy và **macro-F1** (chỉ số chính). Chỉ làm bước này cho baseline và cấu hình cuối cùng.

In [ ]:
# TODO: chọn cấu hình cuối cùng theo VAL (có thể là baseline nếu bạn không kết hợp thêm gì)
# TODO: final_eval(cfg_final, result_final, data, f"{OUT_DIR}/predictions_eval.csv")
# TODO: chạy: python scripts/evaluate.py --pred <OUT_DIR>/predictions_eval.csv --out <OUT_DIR>/eval_result.json (từ REPO_ROOT)
# TODO: đọc eval_result.json: ghi accuracy, macro_f1 vào bảng và báo cáo; làm tương tự cho baseline (predictions riêng, không nộp nếu không cần)

In [ ]:
# TODO: phân tích lỗi trên eval: bảng precision/recall/F1 theo lớp (từ eval_result.json) và ma trận nhầm lẫn;
#       lớp nào khó nhất? vì sao (số mẫu ít? nhầm với lớp nào?)  -> viết nhận xét ở ô markdown bên dưới

**Phân tích lỗi (tự viết):** ...

In [ ]:
# TODO: results = load_results(f"{OUT_DIR}/results"); rows = [to_row(r, ...) for r in results]
# TODO: write_xlsx(rows, f"{REPO_ROOT}/templates/experiment_table_template.xlsx", f"{OUT_DIR}/experiments.xlsx")
# TODO: bổ sung sheet Seeds (exp_id các lần chạy baseline) và nhận xét ở sheet Summary